# Purpose of Support Vector Machine (SVM)

## Overview
**Support Vector Machine (SVM)** is a powerful supervised learning algorithm used for both classification and regression tasks. It finds the optimal hyperplane that maximally separates different classes in the feature space.

## Why Learn SVM?

1. **Effective High-Dimensional Learning:** Performs exceptionally well in high-dimensional spaces where other algorithms struggle

2. **Kernel Trick:** Can handle non-linear separable data through kernel functions (linear, RBF, polynomial, sigmoid)

3. **Margin Maximization:** Focuses on maximizing the margin (distance) between classes, promoting generalization

4. **Practical Applications:**
   - Text classification and sentiment analysis
   - Image recognition and face detection
   - Medical diagnosis and cancer detection
   - Handwriting and digit recognition
   - Bioinformatics and protein classification

5. **Robustness:** Less prone to overfitting compared to neural networks, especially with small datasets

6. **Support Vectors:** Uses only a subset of training data (support vectors) for predictions, making it memory-efficient

## Key Learning Objectives
- ✓ Understand the concept of hyperplanes and margins
- ✓ Learn different kernel functions (linear, RBF, polynomial, sigmoid)
- ✓ Tune critical hyperparameters: C (regularization) and gamma (kernel coefficient)
- ✓ Use GridSearchCV to find optimal SVM parameters
- ✓ Evaluate model performance using confusion matrices and classification metrics
- ✓ Visualize decision boundaries and support vectors
- ✓ Compare SVM with other algorithms (KNN, Random Forest)
- ✓ Recognize when to use SVM and its limitations

## What You'll Build
This notebook demonstrates SVM on the **Iris dataset** - the same dataset used for KNN to allow for easy comparison. You'll:
1. Load and explore the Iris dataset (150 samples, 4 features, 3 classes)
2. Test different kernels (linear, RBF, polynomial) to understand their effects
3. Manually tune C and gamma parameters
4. Use GridSearchCV for comprehensive hyperparameter optimization
5. Evaluate using confusion matrices, accuracy, precision, recall, and F1-scores
6. Visualize 2D decision boundaries
7. Compare performance across different kernels

---

## Step 1: Import Libraries & Load Dataset

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Create a DataFrame for better visualization
iris_df = pd.DataFrame(X_iris, columns=iris.feature_names)
iris_df['Target'] = y_iris
iris_df['Species'] = iris_df['Target'].map({0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'})

print("IRIS DATASET - LOADED")
print("=" * 70)
print(f"Shape: {X_iris.shape}")
print(f"Features: {iris.feature_names}")
print(f"Classes: {iris.target_names}")
print(f"Target: {np.unique(y_iris)}")
print("\nFirst 5 rows:")
print(iris_df.head())
print("\nDataset Statistics:")
print(iris_df.describe())

## Step 2: Train-Test Split & Feature Scaling

**CRITICAL:** SVM is distance-based and sensitive to feature scaling. Features must be scaled to the same range (zero mean, unit variance) for optimal performance.

In [ ]:
# Train-test split with stratification
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

# Feature scaling (MANDATORY for SVM)
scaler_iris = StandardScaler()
X_train_iris_scaled = scaler_iris.fit_transform(X_train_iris)
X_test_iris_scaled = scaler_iris.transform(X_test_iris)

print("DATA SPLIT & SCALING")
print("=" * 70)
print(f"Training Set Size: {X_train_iris_scaled.shape}")
print(f"Test Set Size: {X_test_iris_scaled.shape}")
print(f"\nTraining Set Class Distribution:")
unique, counts = np.unique(y_train_iris, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls} ({iris.target_names[cls]}): {cnt} samples")
print(f"\nFeature Scaling Applied (Mean=0, Std=1)")
print(f"Train - Mean: {X_train_iris_scaled.mean(axis=0).round(3)}")
print(f"Train - Std:  {X_train_iris_scaled.std(axis=0).round(3)}")

## Step 3: Understand SVM - Test Different Kernels Manually

**Kernel Functions:**
- **Linear:** Best for linearly separable data (simplest, fastest)
- **RBF (Radial Basis Function):** Best for non-linear data (most commonly used)
- **Polynomial:** Good for polynomial decision boundaries
- **Sigmoid:** Similar to neural network activation

In [ ]:
# Test different kernels
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
svm_kernel_results = []

print("SVM - KERNEL COMPARISON")
print("=" * 85)
print(f"{'Kernel':<12} {'Train Accuracy':<18} {'Test Accuracy':<18} {'Train F1':<15} {'Test F1':<15}")
print("-" * 85)

for kernel in kernels:
    try:
        svm = SVC(kernel=kernel, random_state=42, C=1.0)
        svm.fit(X_train_iris_scaled, y_train_iris)
        
        # Predictions
        y_pred_train = svm.predict(X_train_iris_scaled)
        y_pred_test = svm.predict(X_test_iris_scaled)
        
        # Metrics
        train_acc = accuracy_score(y_train_iris, y_pred_train)
        test_acc = accuracy_score(y_test_iris, y_pred_test)
        train_f1 = f1_score(y_train_iris, y_pred_train, average='weighted')
        test_f1 = f1_score(y_test_iris, y_pred_test, average='weighted')
        
        svm_kernel_results.append({
            'Kernel': kernel,
            'Train_Accuracy': train_acc,
            'Test_Accuracy': test_acc,
            'Train_F1': train_f1,
            'Test_F1': test_f1,
            'Support_Vectors': len(svm.support_vectors_)
        })
        
        print(f"{kernel:<12} {train_acc:<18.4f} {test_acc:<18.4f} {train_f1:<15.4f} {test_f1:<15.4f}")
    except Exception as e:
        print(f"{kernel:<12} ERROR: {str(e)[:30]}")

svm_kernel_df = pd.DataFrame(svm_kernel_results)
best_kernel = svm_kernel_df.loc[svm_kernel_df['Test_Accuracy'].idxmax(), 'Kernel']
best_acc_kernel = svm_kernel_df.loc[svm_kernel_df['Test_Accuracy'].idxmax(), 'Test_Accuracy']

print(f"\n✓ Best Kernel: {best_kernel} with Test Accuracy: {best_acc_kernel:.4f}")

## Step 4: Manual Parameter Tuning - C (Regularization) Effect

**C Parameter:**
- **Small C (0.001-0.1):** Wider margin, more misclassification tolerance (underfitting risk)
- **Medium C (1.0-10):** Balanced approach (typical default)
- **Large C (100+):** Narrow margin, less tolerance (overfitting risk)

In [ ]:
# Test different C values (RBF kernel)
c_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
svm_c_results = []

print("SVM - C PARAMETER TUNING (RBF Kernel)")
print("=" * 85)
print(f"{'C':<12} {'Train Accuracy':<18} {'Test Accuracy':<18} {'Train F1':<15} {'Test F1':<15}")
print("-" * 85)

for c in c_values:
    svm = SVC(kernel='rbf', C=c, gamma='scale', random_state=42)
    svm.fit(X_train_iris_scaled, y_train_iris)
    
    # Predictions
    y_pred_train = svm.predict(X_train_iris_scaled)
    y_pred_test = svm.predict(X_test_iris_scaled)
    
    # Metrics
    train_acc = accuracy_score(y_train_iris, y_pred_train)
    test_acc = accuracy_score(y_test_iris, y_pred_test)
    train_f1 = f1_score(y_train_iris, y_pred_train, average='weighted')
    test_f1 = f1_score(y_test_iris, y_pred_test, average='weighted')
    
    svm_c_results.append({
        'C': c,
        'Train_Accuracy': train_acc,
        'Test_Accuracy': test_acc,
        'Train_F1': train_f1,
        'Test_F1': test_f1
    })
    
    print(f"{c:<12} {train_acc:<18.4f} {test_acc:<18.4f} {train_f1:<15.4f} {test_f1:<15.4f}")

svm_c_df = pd.DataFrame(svm_c_results)
best_c = svm_c_df.loc[svm_c_df['Test_Accuracy'].idxmax(), 'C']
print(f"\n✓ Best C (RBF): {best_c}")

## Step 5: GridSearchCV - Exhaustive SVM Parameter Optimization

Perform 2D grid search for optimal kernel, C, and gamma parameters using cross-validation.

In [ ]:
# GridSearchCV for SVM
svm_param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]
}

svm_gs = GridSearchCV(SVC(), svm_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
svm_gs.fit(X_train_iris_scaled, y_train_iris)

print("GRIDSEARCHCV - SVM PARAMETER OPTIMIZATION")
print("=" * 80)
print(f"Best Kernel: {svm_gs.best_params_['kernel']}")
print(f"Best C: {svm_gs.best_params_['C']}")
print(f"Best Gamma: {svm_gs.best_params_['gamma']}")
print(f"Best CV Accuracy Score: {svm_gs.best_score_:.4f}")
print(f"Total Combinations Tested: {len(svm_gs.cv_results_['params'])}")
print(f"Total Model Fits (CV): {len(svm_gs.cv_results_['params']) * 5}")

# Evaluate best model on test set
svm_best = svm_gs.best_estimator_
y_pred_svm = svm_best.predict(X_test_iris_scaled)
svm_test_accuracy = accuracy_score(y_test_iris, y_pred_svm)
svm_test_f1 = f1_score(y_test_iris, y_pred_svm, average='weighted')

print(f"\nTest Set Performance (Best SVM):")
print(f"Test Accuracy: {svm_test_accuracy:.4f}")
print(f"Test F1 Score: {svm_test_f1:.4f}")
print(f"Number of Support Vectors: {len(svm_best.support_vectors_)} out of {len(X_train_iris_scaled)}")

## Step 6: Model Evaluation - Confusion Matrix & Classification Report

In [ ]:
# Detailed evaluation
cm = confusion_matrix(y_test_iris, y_pred_svm)
cr = classification_report(y_test_iris, y_pred_svm, target_names=iris.target_names)

print("CONFUSION MATRIX")
print("=" * 60)
print(cm)

print("\n\nCLASSIFICATION REPORT")
print("=" * 60)
print(cr)

# Cross-validation scores
cv_scores = cross_val_score(svm_best, X_train_iris_scaled, y_train_iris, cv=5, scoring='accuracy')
print(f"\nCROSS-VALIDATION SCORES (5-Fold):")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

## Step 7: Visualization - Kernel & C Parameter Effects

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('SVM Classification - Iris Dataset Analysis', fontsize=16, fontweight='bold')

# Plot 1: Kernel comparison
ax1 = axes[0, 0]
x_pos = np.arange(len(svm_kernel_df))
width = 0.35
ax1.bar(x_pos - width/2, svm_kernel_df['Train_Accuracy'], width, label='Train Accuracy', edgecolor='black')
ax1.bar(x_pos + width/2, svm_kernel_df['Test_Accuracy'], width, label='Test Accuracy', edgecolor='black')
ax1.set_xlabel('Kernel', fontsize=11, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax1.set_title('SVM: Accuracy by Kernel Type', fontsize=12, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(svm_kernel_df['Kernel'])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim([0.9, 1.0])

# Plot 2: C parameter effect
ax2 = axes[0, 1]
ax2.semilogx(svm_c_df['C'], svm_c_df['Train_Accuracy'], 'o-', label='Train Accuracy', linewidth=2, markersize=8)
ax2.semilogx(svm_c_df['C'], svm_c_df['Test_Accuracy'], 's-', label='Test Accuracy', linewidth=2, markersize=8)
ax2.axvline(best_c, color='red', linestyle='--', alpha=0.7, label=f'Best C={best_c}')
ax2.set_xlabel('C (Regularization)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax2.set_title('SVM: Accuracy vs C Parameter', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0.85, 1.0])

# Plot 3: Confusion Matrix Heatmap
ax3 = axes[1, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=ax3,
            xticklabels=iris.target_names, yticklabels=iris.target_names,
            annot_kws={'size': 12, 'weight': 'bold'})
ax3.set_ylabel('True Label', fontsize=11, fontweight='bold')
ax3.set_xlabel('Predicted Label', fontsize=11, fontweight='bold')
ax3.set_title(f'Confusion Matrix (Kernel={svm_gs.best_params_["kernel"]}, C={svm_gs.best_params_["C"]})', 
              fontsize=12, fontweight='bold')

# Plot 4: Support Vectors Analysis
ax4 = axes[1, 1]
support_vector_counts = svm_kernel_df['Support_Vectors'].values
kernels_list = svm_kernel_df['Kernel'].values
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = ax4.bar(kernels_list, support_vector_counts, color=colors, edgecolor='black', linewidth=1.5)
ax4.set_ylabel('Number of Support Vectors', fontsize=11, fontweight='bold')
ax4.set_xlabel('Kernel', fontsize=11, fontweight='bold')
ax4.set_title('Support Vectors per Kernel', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')
for bar, count in zip(bars, support_vector_counts):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(count)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## Step 8: SVM Summary & Key Takeaways

# Support Vector Machine (SVM) - Complete Summary

## 1. What is SVM?
**Support Vector Machines** are supervised learning algorithms that find the optimal hyperplane maximizing the margin between different classes. The "support vectors" are the data points closest to the decision boundary.

**Core Principle:** Maximize margin = Better generalization

## 2. Key Concepts

### Hyperplane
- Decision boundary separating classes
- In 2D: a line; in 3D: a plane; in 4D+: a hyperplane

### Margin
- Distance from hyperplane to nearest data points
- SVM maximizes this margin for better separation
- Wider margin = better generalization

### Support Vectors
- Training points closest to the decision boundary
- Define the hyperplane
- Only these points matter for predictions (memory efficient)

## 3. Kernel Functions

| Kernel | Formula | Best For | Complexity |
|--------|---------|----------|------------|
| **Linear** | w·x + b | Linearly separable data | Low |
| **RBF** | exp(-γ\|x-x'\|²) | Non-linear, multi-class | Medium |
| **Polynomial** | (γ·x·x' + r)^d | Polynomial decision boundaries | Medium-High |
| **Sigmoid** | tanh(γ·x·x' + r) | Neural network-like | Low |

## 4. Critical Hyperparameters

### C (Regularization Parameter)
- **Small C (0.001-0.1):** Larger margin, more tolerance for misclassification (underfitting)
- **Default C=1:** Balanced approach
- **Large C (100+):** Smaller margin, less tolerance (overfitting)

### Gamma (Kernel Coefficient)
- Defines influence range of single training example
- **Low gamma:** Far-reaching influence (smooth decision boundary)
- **High gamma:** Local influence (complex decision boundary)
- Only relevant for RBF, polynomial, sigmoid kernels

## 5. Advantages of SVM
✓ Excellent for high-dimensional data  
✓ Effective when features > samples  
✓ Memory efficient (uses only support vectors)  
✓ Robust to outliers with proper C tuning  
✓ Flexible with multiple kernel options  
✓ Works well for binary and multi-class problems  

## 6. Disadvantages of SVM
✗ Slow training on large datasets (O(n²) or O(n³))  
✗ Not suitable for very large datasets (100K+ samples)  
✗ Hyperparameter tuning critical for good performance  
✗ Feature scaling mandatory  
✗ Difficult to interpret predictions (black box)  
✗ Not suitable for imbalanced datasets without class weights  

## 7. When to Use SVM
- Small to medium datasets (< 100K samples)
- High-dimensional data (many features)
- Binary classification
- Text classification and NLP tasks
- Image recognition
- Need interpretable decision boundaries

## 8. When NOT to Use SVM
- Very large datasets (use SGDClassifier instead)
- Real-time predictions needed (training too slow)
- Highly imbalanced data (without careful weighting)
- Need probabilistic outputs (SVC doesn't provide them by default)

## 9. SVM vs Other Algorithms

| Aspect | SVM | KNN | Random Forest | Neural Net |
|--------|-----|-----|---------------|------------|
| **Speed (Train)** | Slow | Zero | Medium | Slow |
| **Speed (Predict)** | Medium | Slow | Fast | Fast |
| **High Dimensions** | Excellent | Poor | Good | Excellent |
| **Non-linear** | Excellent (kernels) | Good | Excellent | Excellent |
| **Interpretability** | Low | High | Medium | Low |
| **Tuning Effort** | High | Low | Medium | Very High |
| **Large Data** | Poor | Poor | Good | Excellent |

## 10. Hyperparameter Tuning Workflow
1. **Data Preprocessing:** Mandatory feature scaling (StandardScaler)
2. **Kernel Selection:** Test linear, RBF, poly (RBF usually best)
3. **Manual Testing:** Understand C and gamma effects
4. **GridSearchCV:** Comprehensive parameter search with CV
5. **Evaluation:** Confusion matrix, classification report, CV scores
6. **Validation:** Test on separate hold-out set

## 11. Best Practices
1. **Always scale features** - Use StandardScaler
2. **Start with RBF kernel** - Most versatile for non-linear problems
3. **Use GridSearchCV** - Don't manually tune parameters
4. **Cross-validation is critical** - Use 5-fold or 10-fold CV
5. **Handle imbalanced data** - Use class_weight='balanced'
6. **Set probability=True** - If you need confidence scores (adds computation)

## 12. Common Mistakes
❌ Forgetting feature scaling → Wrong predictions  
❌ Using default parameters → Suboptimal performance  
❌ No cross-validation → Overestimated accuracy  
❌ Using SVM on hundreds of thousands of samples → Too slow  
❌ Not balancing classes → Biased toward majority class  
❌ Ignoring support vector count → May indicate overfitting  

---

**Next Steps:**
- Try SVM on more complex datasets (MNIST, CIFAR, text data)
- Compare SVM with Random Forest and Neural Networks
- Implement One-vs-One and One-vs-Rest for multi-class
- Explore probability calibration (CalibratedClassifierCV)
- Use SVM for regression (SVR - Support Vector Regression)